# Kiểm tra chất lượng annotation

Notebook kiểm tra annotation YOLO của tập PKU-Market-PCB trước khi tạo split và huấn luyện. Báo cáo này là bằng chứng kiểm soát chất lượng dữ liệu: mọi lỗi được ghi vào `results/annotation_issues.csv`; nếu không có lỗi, CSV vẫn được xuất với hàng tiêu đề.

Các kiểm tra gồm: cấu trúc ảnh-nhãn, `class_id`, tọa độ chuẩn hóa, box vượt biên, box rỗng, box trùng hoàn toàn và định dạng nhãn không hợp lệ. Notebook chỉ đọc dữ liệu gốc trong `data/raw/`.

In [1]:
import csv
import math
from pathlib import Path


IMAGE_SUFFIXES = {".bmp", ".jpeg", ".jpg", ".png", ".tif", ".tiff", ".webp"}
CSV_COLUMNS = [
    "issue_type",
    "image_path",
    "label_path",
    "line_number",
    "class_id",
    "x_center",
    "y_center",
    "width",
    "height",
    "details",
]


def project_root() -> Path:
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / "data").is_dir() and (candidate / "results").is_dir():
            return candidate
    raise RuntimeError("Không tìm thấy thư mục gốc của dự án.")

## Ghép ảnh với file nhãn

Dataset dùng cấu trúc YOLO `images/` và `labels/`. Một phần ảnh tăng cường nằm trong thư mục con, trong khi file nhãn tương ứng được lưu phẳng trong `labels/`; hàm ghép nhãn xử lý cả hai cấu trúc này để không ghi nhận nhầm lỗi thiếu nhãn.

In [2]:
def label_for_image(image_path: Path, dataset_dir: Path) -> Path:
    relative = image_path.relative_to(dataset_dir)
    if "images" in relative.parts:
        index = relative.parts.index("images")
        labels_dir = dataset_dir.joinpath(*relative.parts[:index], "labels")
        mirrored_path = labels_dir.joinpath(
            *relative.parts[index + 1:]
        ).with_suffix(".txt")
        flattened_path = labels_dir / image_path.with_suffix(".txt").name
        # Một số ảnh tăng cường nằm trong thư mục con, còn nhãn được lưu phẳng.
        return next((path for path in (mirrored_path, flattened_path) if path.is_file()), mirrored_path)
    return image_path.with_suffix(".txt")


def image_for_label(label_path: Path, dataset_dir: Path) -> Path | None:
    relative = label_path.relative_to(dataset_dir)
    if "labels" in relative.parts:
        index = relative.parts.index("labels")
        image_base = dataset_dir.joinpath(
            *relative.parts[:index], "images", *relative.parts[index + 1:]
        ).with_suffix("")
        return next(
            (image_base.with_suffix(suffix) for suffix in IMAGE_SUFFIXES if image_base.with_suffix(suffix).is_file()),
            None,
        )
    image_base = label_path.with_suffix("")
    return next(
        (image_base.with_suffix(suffix) for suffix in IMAGE_SUFFIXES if image_base.with_suffix(suffix).is_file()),
        None,
    )


def relative_path(path: Path | None, dataset_dir: Path) -> str:
    return str(path.relative_to(dataset_dir)) if path else ""


def issue(issue_type: str, dataset_dir: Path, **values: object) -> dict[str, object]:
    row: dict[str, object] = {column: "" for column in CSV_COLUMNS}
    row["issue_type"] = issue_type
    row.update(values)
    for field in ("image_path", "label_path"):
        if isinstance(row[field], Path):
            row[field] = relative_path(row[field], dataset_dir)
    return row

## Kiểm tra từng bounding box

Mỗi dòng nhãn phải theo định dạng YOLO: `class_id x_center y_center width height`. Tọa độ phải nằm trong `[0, 1]`; đồng thời biên trái, phải, trên và dưới của box không được vượt ảnh. Các box có cùng class và bốn tọa độ giống hệt nhau trong một ảnh được coi là trùng hoàn toàn.

In [3]:
def validate_label(label_path: Path, image_path: Path, dataset_dir: Path) -> list[dict[str, object]]:
    issues: list[dict[str, object]] = []
    seen_boxes: dict[tuple[int, float, float, float, float], int] = {}

    for line_number, line in enumerate(label_path.read_text(encoding="utf-8").splitlines(), start=1):
        if not line.strip():
            continue
        fields = line.split()
        if len(fields) != 5:
            issues.append(
                issue(
                    "malformed_label",
                    dataset_dir,
                    image_path=image_path,
                    label_path=label_path,
                    line_number=line_number,
                    details="YOLO label phải có đúng 5 giá trị.",
                )
            )
            continue

        try:
            class_id = int(fields[0])
            x_center, y_center, width, height = (float(value) for value in fields[1:])
        except ValueError:
            issues.append(
                issue(
                    "malformed_label",
                    dataset_dir,
                    image_path=image_path,
                    label_path=label_path,
                    line_number=line_number,
                    details="class_id phải là số nguyên; tọa độ phải là số thực.",
                )
            )
            continue

        values = {
            "image_path": image_path,
            "label_path": label_path,
            "line_number": line_number,
            "class_id": class_id,
            "x_center": x_center,
            "y_center": y_center,
            "width": width,
            "height": height,
        }
        coordinates = (x_center, y_center, width, height)
        if class_id not in range(6):
            issues.append(issue("class_id_out_of_range", dataset_dir, **values, details="class_id phải nằm trong khoảng 0-5."))
        if not all(math.isfinite(value) and 0 <= value <= 1 for value in coordinates):
            issues.append(issue("coordinate_outside_unit_interval", dataset_dir, **values, details="Tọa độ YOLO phải nằm trong [0, 1]."))
        if width == 0 or height == 0:
            issues.append(issue("empty_box", dataset_dir, **values, details="Width và height phải lớn hơn 0."))
        if all(math.isfinite(value) for value in coordinates):
            x_min, x_max = x_center - width / 2, x_center + width / 2
            y_min, y_max = y_center - height / 2, y_center + height / 2
            if x_min < 0 or x_max > 1 or y_min < 0 or y_max > 1:
                issues.append(issue("box_outside_image", dataset_dir, **values, details="Box vượt biên ảnh sau khi quy đổi tọa độ YOLO."))

        box = (class_id, x_center, y_center, width, height)
        if box in seen_boxes:
            issues.append(
                issue(
                    "duplicate_box",
                    dataset_dir,
                    **values,
                    details=f"Trùng hoàn toàn với box ở dòng {seen_boxes[box]}.",
                )
            )
        else:
            seen_boxes[box] = line_number
    return issues

## Kiểm tra toàn bộ tập dữ liệu

Tất cả ảnh và file nhãn được đối chiếu hai chiều. Một file nhãn dùng chung cho ảnh tăng cường chỉ được kiểm tra nội dung một lần, tránh lặp cùng một lỗi trong báo cáo.

In [4]:
def validate(dataset_dir: Path) -> list[dict[str, object]]:
    image_paths = sorted(
        path for path in dataset_dir.rglob("*") if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
    )
    label_paths = sorted(path for path in dataset_dir.rglob("*.txt") if path.name != "Class.txt")
    issues: list[dict[str, object]] = []
    matched_labels: set[Path] = set()
    validated_labels: set[Path] = set()

    for image_path in image_paths:
        label_path = label_for_image(image_path, dataset_dir)
        if not label_path.is_file():
            issues.append(issue("image_without_label", dataset_dir, image_path=image_path, details="Không tìm thấy file nhãn tương ứng."))
            continue
        matched_labels.add(label_path)
        if label_path not in validated_labels:
            issues.extend(validate_label(label_path, image_path, dataset_dir))
            validated_labels.add(label_path)

    for label_path in label_paths:
        if label_path not in matched_labels and image_for_label(label_path, dataset_dir) is None:
            issues.append(issue("label_without_image", dataset_dir, label_path=label_path, details="Không tìm thấy ảnh tương ứng."))
    return issues

## Xuất báo cáo

CSV luôn được tạo. Khi không có lỗi, file chỉ gồm header, qua đó chứng minh rằng quy trình kiểm tra đã được chạy thay vì thiếu báo cáo.

In [5]:
ROOT = project_root()
DATASET_DIR = ROOT / "data/raw/PKU-Market-PCB(Data enhanced version)"
OUTPUT_PATH = ROOT / "results/annotation_issues.csv"

if not DATASET_DIR.is_dir():
    raise FileNotFoundError(f"Không tìm thấy thư mục dataset: {DATASET_DIR}")

issues = validate(DATASET_DIR)
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_PATH.open("w", newline="", encoding="utf-8") as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=CSV_COLUMNS)
    writer.writeheader()
    writer.writerows(issues)
print(f"Đã kiểm tra {DATASET_DIR} và ghi {len(issues)} lỗi vào {OUTPUT_PATH}.")

Đã kiểm tra /Users/Project/kltn-pcb/data/raw/PKU-Market-PCB(Data enhanced version) và ghi 0 lỗi vào /Users/Project/kltn-pcb/results/annotation_issues.csv.
